### Anleitung, um mit der assign()-Funktion aus SCALOP zu arbeiten

__1: Projekt klonen__

```bash
git clone https://github.com/oxpig/SCALOP.git
```

__2: Conda-Umgebung erstellen__

```bash
conda create -n scalop-env python=3.8 -y
```

__3.1: Conda initialisieren (falls noch nie gemacht)__

```bash
conda init
```

__3.2: dann Terminal schließen__

```bash
exit
```

__3.3: dann Terminal neu öffnen__

__4: Umgebung aktivieren__

```bash
conda activate scalop-env
```

__5: Abhängigkeiten installieren__ (hmmer benötigt Linux oder macOS)

```bash
conda install -c bioconda numpy biopython -y
conda install -c bioconda hmmer -y
```

__6: SCALOP lokal installieren__

```bash
pip install ./SCALOP
```

__7: Kernel für Jupyter registrieren__

```bash
pip install ipykernel
python -m ipykernel install --user --name scalop-env --display-name "Python (scalop-env)"
```

__8: Dann Jupyter Notebook öffnen und den Kernel "Python (scalop-env)" auswählen. Jetzt kann man SCALOP in Jupyter Notebooks verwenden.__

In [2]:
#from scalop.predict import assign
import pandas as pd
import numpy as np

df = pd.read_csv("data/ab_ag_vseqs.tsv", sep="\t")
regions = ["H1", "H2", "L1", "L2", "L3"]
seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]

In [5]:
# Zusammenfassung: Auf jede Sequenz der variablen Domänen (VH, VL) wird die assign()-Funktion angewendet, um die CDR-Sequenzen nach Chothia (für unser Clustering über naiven Ansatz, ESM-C und MMSeq2
# und die Zuordnung in kanonische Cluster (für Vergleich mit Ground Truth) zu extrahieren. Input-Tabelle ab_ag_vseqs.tsv wird erweitert um 5 Spalten für CDR-Sequenzen und 5 Spalten für Cluster-Zuordnung (je eine pro CDR).

fab_lists = df[["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species", "VH", "VL"]].values.tolist()
# df[...] --> gibt pd.DataFrame mit den ausgewählten Spalten aus
# .values --> wandelt pd.DataFrame in np.ndarray aus, weil .tolist() nicht auf pd.DataFrames angwendet werden kann
# .tolist() --> wandelt np.ndarray in eine Liste an Listen um (übersichtlicher zu indizieren als np.array und schneller verarbeitet als pd.DataFrame)

df_rows = [] # Bereitet äußere Liste für Liste an Listen vor. Innere Listen entsprechen später den Zeilen des neuen pd.DataFrames

for fab_list in fab_lists: # fab_list Name wurde aus sabdab_downloader.py zur Übersichtlichkeit übernommen. Eine fab_list entspricht einer Zeile in ab_ag_vseqs.tsv
    pdb = fab_list[0]
    hchain = fab_list[1]
    lchain = fab_list[2]
    model = fab_list[3]
    antigen_name = fab_list[4]
    antigen_species = fab_list[5]
    vh_seq = fab_list[6]
    vl_seq = fab_list[7]
    # Extrahiert alle relevanten Informationen aus einer Zeile

    try:
        results_vh = assign(vh_seq, scheme="chothia", definition="chothia")[0]['outputs']
        results_vl = assign(vl_seq, scheme="chothia", definition="chothia")[0]['outputs']
        # vollständiger Input für assign(): [(seqname_H, seq_H), (seqname_L, seq_L)]
        # optional: nur einer von seq_H bzw. seq_L reicht; seqname = "input" wenn nur kein seqname angegeben wird
        # vollständiger Output von assign(): [{dict_H}, {dict_L}] (list); ein dict entfällt ggf, wenn nur eine Sequenz angegeben
        # Aufbau von dict_H (dict_L analog):
        # {
        # key seqname (str) : value input (str)
        # key input (str) : value (input (str), AS-Sequenz (str))
        # key outputs (str) : value {"H1" : ["H1", AS-Sequenz (str), Cluster-Name (str), f"{pdb}_{chain_ID}" für ähnlichste Kette in der SAbDab], "H2" : [...]}
        # }
        # assign() auf VH- und VL-Sequenz anwenden (Zählweise (scheme) und CDR-Grenzen (definition) nach Chothia, nicht IMGT oder North)
        # assign() nimmt als Input eine VH bzw. VL Sequenz, oder beide getrennt durch ein Komma
        # -->
        # liste mit einem dictionary (eins für VH und eins für VL ?)
        # dictionary weist jeweils zu:
        # key "seqname" (str) : value "input" (str)
        # key "input" (str) : value 
        # ==> 
        # key "outputs" (str) : {
        #   *für jede Region, z.B.:* "L1" (str) : 
        # } (dict)
        seqs_and_cfs = []
        # Bereitet Liste für die Spalten "SEQ_H1", ..., "SEQ_L3" (Sequenzen) und "CF_H1", ... "CF_L3" (Canonical Forms) vor
        for region in regions:
            chain_type = region[0]  #z.B. "H" aus "H1" oder "L" aus "L1".
            
            if chain_type == "H":
                seq = results_vh[region][1]
                cf = results_vh[region][2]
            else:
                seq = results_vl[region][1]
                cf = results_vl[region][2]
            # Wählt Ergebnisse für SEQ und CF entweder für VH bzw VL aus

            seqs_and_cfs.append(seq)
            seqs_and_cfs.append(cf)
            # In einem Durchlauf erst SEQ und CF der entsprechenden Region auswählen
            # Nach Durchlauf aller Regionen: "SEQ_H1", "CF_H1", ..., "SEQ_L3", "CF_L3"
        df_row = [pdb, hchain, lchain, model, antigen_name, antigen_species] + seqs_and_cfs #df_row entspricht einer Reihe des zu erstellendes pd.Dataframes
    
    except Exception as e:
        print(f"Fehler bei {pdb}: {e}")
        df_row = [pdb, hchain, lchain, model, antigen_name, antigen_species] + [np.nan]*10
        # Wenn ein Fehler auftritt (z.B. bei assign()) --> df_row enthält für alle SEQ und CF Spalten np.nan --> Zeile wird später verworfen
    
    df_rows.append(df_row) # Ergänzt äußere Liste bei jedem Durchlauf um eine Liste df_row, d.h. um eine Reihe des späteren pd.DataFrames

ab_ag_scalop = (
    pd.DataFrame(df_rows, columns=["pdb", "Hchain", "Lchain", "model", "antigen_name", "antigen_species", "SEQ_H1", "CF_H1", "SEQ_H2", "CF_H2", "SEQ_L1", "CF_L1", "SEQ_L2", "CF_L2", "SEQ_L3", "CF_L3"])
    .dropna(subset = seq_regions)  # Verwerfe  Zeilen, in denen mindestens eine CDR-Sequenz fehlt
    .reset_index(drop = True)
)
# Erstellt einen pd.DataFrame aus einer Liste an Listen
# #.drop_duplicates(subset = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]) wird noch nicht angewendet, weil ab_ag_scalop.tsv für die Embeddings für
# den ESM-C-Ansatz benötigt wird --> Embeddings sollen zur Sicherheit für alle Sequenzen erstellt werden, damit uns keins fehlt

    
ab_ag_scalop.to_csv('data/ab_ag_scalop.tsv', sep='\t', index=False)
# Speichert den pd.DataFrame als .tsv im Ornder data. Index-Spalte aus dem pd.DataFrame soll nicht als Spalte in die .tsv übernommen werden

Fehler bei 8uzp: name 'assign' is not defined
Fehler bei 8uzp: name 'assign' is not defined
Fehler bei 8veb: name 'assign' is not defined
Fehler bei 8veb: name 'assign' is not defined
Fehler bei 8veb: name 'assign' is not defined
Fehler bei 8ved: name 'assign' is not defined
Fehler bei 8ved: name 'assign' is not defined
Fehler bei 8ved: name 'assign' is not defined
Fehler bei 8vee: name 'assign' is not defined
Fehler bei 8vee: name 'assign' is not defined
Fehler bei 8vee: name 'assign' is not defined
Fehler bei 8vef: name 'assign' is not defined
Fehler bei 8vef: name 'assign' is not defined
Fehler bei 8vef: name 'assign' is not defined
Fehler bei 8vpf: name 'assign' is not defined
Fehler bei 8vpf: name 'assign' is not defined
Fehler bei 8vpf: name 'assign' is not defined
Fehler bei 9dpc: name 'assign' is not defined
Fehler bei 9l6c: name 'assign' is not defined
Fehler bei 9l6c: name 'assign' is not defined
Fehler bei 9l6c: name 'assign' is not defined
Fehler bei 9mer: name 'assign' is 

In [ ]:
# Diese Schritte müssen beim Aufrufen von ab_ag_scalop.tsv durchgeführt werden:
import pandas as pd
seq_regions = ["SEQ_H1", "SEQ_H2", "SEQ_L1", "SEQ_L2", "SEQ_L3"]
cf_regions = ["CF_H1", "CF_H2", "CF_L1", "CF_L2", "CF_L3"]
df = (
    pd.read_csv("data/ab_ag_scalop.tsv", sep="\t")
    .dropna(subset = seq_regions)
    .dropna(subset = cf_regions)
    .drop_duplicates(subset = seq_regions) 
)

antigen_counts = df["antigen_name"].value_counts() # Tabelle aus antigen_names und ihren Häufigkeiten in der Spalte antigen_name
df = df[df["antigen_name"].isin(antigen_counts[antigen_counts >= 5].index)] # Behält nur Zeilen, deren antigen_name mindestens 5-mal vorkommt
